# Nutrition5k: resultados de los híbridos móviles

No entrena nada: lee los runs de `outputs/hybrid_mobile/` (los escribe `experiments.ipynb` de esta carpeta) con las
funciones de `src/nutrition5k/results.py`. Elegir siempre por **validación** (`val_mean_pmae`); test se informa pero no
decide. `test_clean_mean_pmae` es el test oficial sin los platos de `data/nutrition5k/excluded_dishes.csv`.

La pareja `v4m` ↔ `v4hm` (Q2a) se compara en `notebooks/results_general.ipynb` o cargando también `mobilenet`.

In [ ]:
import os
from pathlib import Path

if Path("/workspace").is_dir():  # RunPod
    # The clone made by experiments.ipynb: outputs/ links to the volume and data/ holds the images.
    os.chdir("/root/thesis-multimodal-health-ai")
elif Path.cwd().parent.name == "notebooks":
    os.chdir("../..")

import pandas as pd

from nutrition5k.results import (
    baseline_row, compare, cost, gallery, load_runs, seed_files, seed_groups, seed_spread, seed_table,
    seed_targets, select, steps, target_curves, target_table, test_metrics,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

FAMILY = "hybrid_mobile"
SPLIT = "pilot"         # "full" o "pilot"

runs = load_runs([FAMILY], SPLIT)
assert len(runs), f"no runs of {FAMILY} in {SPLIT}; train them with experiments.ipynb"
BASELINE = baseline_row(SPLIT)
print(f"{len(runs)} runs; baseline mean PMAE: "
      f"val {BASELINE['val_mean_pmae'].iloc[0]}, test {BASELINE['test_mean_pmae'].iloc[0]}")
runs

## Etapa 1 (L1): arquitecturas (fine-tuning completo, cabeza `single`, 512, 224 px, lr del backbone 1e-4, semilla 0)

Resumen (`best_epoch` es la época con el menor `val_mean_pmae`, la que se guarda en `best.pt` y se evalúa), tabla de
validación con PMAE, MAE, RMSE y R² por objetivo (en verde el mejor de cada columna; MAE y RMSE en g o kcal), PMAE por
objetivo en barras y curvas de validación (el círculo grande marca la mejor época). La fila `baseline` es la línea
base de la media (predice la media de train para todos los platos) sobre la misma validación.

In [ ]:
STAGE1 = dict(heads="single", hidden="512", image_size=224, backbone_lr=1e-4, seed=0)
stage1 = select(runs, unfreeze="all", **STAGE1)
stage1 = stage1.assign(model=stage1["backbone"] + " ft " + stage1["unfreeze"])
compare(stage1, "model", f"{FAMILY}: stage 1", baseline=BASELINE)

### Etapa 1: coste computacional

- `params_M`: parámetros en millones (backbone, tronco y cabezas). `size_MB`: tamaño de `best.pt` (float32).
- `MMACs`: millones de multiplicaciones-acumulaciones por imagen a la resolución del run; no depende del hardware.
- `cpu_ms`: latencia mediana de una imagen (batch 1) en la CPU de esta máquina, con pesos aleatorios. Solo compara
  backbones entre sí en la misma máquina; la latencia que cuenta es la del dispositivo.
- `gpu_memory_gb` y `sec_per_epoch`: pico de memoria de GPU y segundos por época **del entrenamiento**.

In [ ]:
cost(stage1, "model")

## Etapa 2 (L2): ablaciones

Cada finalista sigue su propio camino greedy (mismos `BACKBONES` y `BEST_*` que en `experiments.ipynb`).
Cada paso compara el valor del factor dentro del backbone; menos de ~1 punto es empate.

In [ ]:
BACKBONES = ("fastvit_sa12",)
BEST_UNFREEZE = {"fastvit_sa12": "2"}  # como texto, igual que en experiments.ipynb

for backbone in BACKBONES:
    steps(runs, backbone, {
        "Step 1: fine-tuning depth": ("unfreeze", dict(heads="single", hidden="512", backbone_lr=1e-4, seed=0)),
        "Step 2: backbone learning rate": ("backbone_lr", dict(unfreeze=BEST_UNFREEZE[backbone], heads="single", hidden="512", seed=0)),
    })

## Error por objetivo (test) de todas las ejecuciones terminadas

Ordenadas por validación. Solo se informa: no sirve para elegir.

In [ ]:
target_table(runs[runs["status"] == "done"], "run", split="test")

## Evolución por objetivo de una ejecución

Comprueba si un objetivo empeora mientras la media mejora. La línea negra es la media; la vertical marca la época elegida.

In [ ]:
BEST = runs[runs["status"] == "done"].iloc[0]  # el mejor por validación; o runs[runs["run"] == "<nombre>"].iloc[0]
target_curves(BEST["path"])

## Etapa 3 (L3): media ± SD entre semillas

Una fila por configuración con más de una semilla; gráficas con el error de cada semilla (puntos) y la media ± SD.

In [ ]:
seed_runs = seed_groups(runs)
if len(seed_runs):
    display(seed_table(seed_runs))
    seed_spread(seed_runs)
    seed_targets(seed_runs, "val")
    seed_targets(seed_runs, "test")
else:
    print("no configuration with more than one seed yet")

## Métricas de test de los modelos finales

MAE, PMAE, RMSE y R² por objetivo, con la línea base como referencia. Por defecto, los tres mejores runs por
validación. Debajo, la media ± SD que escribe `train --all-seeds`.

In [ ]:
FINAL = runs[runs["status"] == "done"].head(3)  # o select(runs, backbone=..., unfreeze=...)
test_metrics(FINAL, SPLIT).round(2)

In [ ]:
seed_files(FAMILY, SPLIT)

## Galería de platos de test

Los platos con menor error, los de mayor error y algunos al azar de un run, con su imagen cenital, el valor real y
el predicho. Necesita las imágenes en `data/`.

In [ ]:
gallery(BEST["path"], SPLIT)